**Please set up your credentials JSON as GCP_CREDENTIALS secrets**

In [1]:
import os

os.environ["DESTINATION_CREDENTIALS"] = "/home/david/.ssh/project-a44b4f29-ed58-4b15-810-15282952039f.json" 
os.environ["BUCKET_URL"] = "gs://taxi-rides-ny-82"

In [2]:
%%capture
!pip install dlt[bigquery, gs] # Install for production

In [1]:
%%capture
!pip install dlt[duckdb]# Install for testing

In [3]:
import dlt
import requests
import pandas as pd
from dlt.destinations import filesystem
from io import BytesIO

Ingesting parquet files to GCS.

In [4]:
# Define a dlt source to download and process Parquet files as resources
@dlt.source(name="rides")
def download_parquet():
    prefix = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata"
    for month in range(1, 7):
        file_name = f"yellow_tripdata_2024-0{month}.parquet"
        url = f"{prefix}_2024-0{month}.parquet"
        response = requests.get(url)

        df = pd.read_parquet(BytesIO(response.content))

        # Return the dataframe as a dlt resource for ingestion
        yield dlt.resource(df, name=file_name)


# Initialize the pipeline
pipeline = dlt.pipeline(
    pipeline_name="rides_pipeline",
    destination=filesystem(layout="{schema_name}/{table_name}.{ext}"),
    dataset_name="rides_dataset",
)

# Run the pipeline to load Parquet data into DuckDB
load_info = pipeline.run(download_parquet(), loader_file_format="parquet")

# Print the results
print(load_info)


PipelineStepFailed: Pipeline execution failed at `step=sync` with exception:

<class 'dlt.common.configuration.exceptions.ConfigFieldMissingException'>
Missing 1 field(s) in configuration `FilesystemDestinationClientConfiguration`: `credentials`
field `credentials` is itself a configuration `GcpServiceAccountCredentials` with missing field(s): `project_id`, `private_key`, `client_email` (Union 1 of 2)
  for field `project_id` the following (config provider, key) were tried in order:
    (Environment Variables, RIDES_PIPELINE__DESTINATION__FILESYSTEM__CREDENTIALS__PROJECT_ID)
    (Environment Variables, RIDES_PIPELINE__DESTINATION__CREDENTIALS__PROJECT_ID)
    (Environment Variables, RIDES_PIPELINE__CREDENTIALS__PROJECT_ID)
    (Environment Variables, DESTINATION__FILESYSTEM__CREDENTIALS__PROJECT_ID)
    (Environment Variables, DESTINATION__CREDENTIALS__PROJECT_ID)
    (Environment Variables, CREDENTIALS__PROJECT_ID)
  for field `private_key` the following (config provider, key) were tr

Ingesting data to Database

In [ ]:
# Define a dlt resource to download and process Parquet files as single table
@dlt.resource(name="rides", write_disposition="replace")
def download_parquet():
    prefix = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata'

    for month in range(1, 7):
        url = f"{prefix}_2024-0{month}.parquet"
        response = requests.get(url)

        df = pd.read_parquet(BytesIO(response.content))

        yield df


# Initialize the pipeline
pipeline = dlt.pipeline(
    pipeline_name="rides_pipeline",
    destination="duckdb",  # Use DuckDB for testing
    # destination="bigquery",  # Use BigQuery for production
    dataset_name="rides_dataset",
)

# Run the pipeline to load Parquet data into DuckDB
info = pipeline.run(download_parquet)

# Print the results
print(info)


In [ ]:
import duckdb

conn = duckdb.connect(f"{pipeline.pipeline_name}.duckdb")

# Set search path to the dataset
conn.sql(f"SET search_path = '{pipeline.dataset_name}'")

# Describe the dataset to see loaded tables
res = conn.sql("DESCRIBE").df()
print(res)

In [ ]:
# provide a resource name to query a table of that name
with pipeline.sql_client() as client:
    with client.execute_query(f"SELECT count(1) FROM rides") as cursor:
        data = cursor.df()
print(data)